# 오늘춥나?? — 안양대 500m 격자 자료 모으기 (구글 드라이브 자동 저장)

기상청 API허브 > 고해상도 격자자료 조회(500m) > **1.3 특정지점 다중요소**로 안양대 위치의 시간별 기온·습도·풍속을 모아요.

- **한 달씩** 받아서 구글 드라이브 `내 드라이브/onulchupna/grid/grid_YYYY_MM.csv`로 저장해요.
- 이미 저장된 달은 **건너뛰어요.** Colab이 끊겨도 다시 실행하면 이어서 받아요.
- 이 API는 한 번에 60분까지만 조회돼서, 두 시각씩 묶어 한 달 = 약 360번 호출해요. 받는 동안 Colab 탭을 열어두세요.

**순서:** ① 드라이브 연결·설정 → ② 응답 확인 → ③ 조회 함수 → ④ 월별 수집 → ⑤ 저장된 파일 확인

## ① 구글 드라이브 연결과 설정

실행하면 구글 계정 권한을 묻는 창이 떠요. 허용하면 드라이브가 연결돼요.

In [ ]:
import os, time
import requests
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime, timedelta
from google.colab import drive

drive.mount("/content/drive")
candidates = [p for p in Path("/content/drive/MyDrive").rglob("perceived-temperature-forAyu")
              if (p / "prepare_training_data.py").is_file()]
if len(candidates) != 1:
    raise RuntimeError(f"프로젝트 폴더를 하나만 찾을 수 있어야 합니다. 검색 결과: {candidates}")
PROJECT_DIR = str(candidates[0])
GRID_DIR = f"{PROJECT_DIR}/data/grid"
os.makedirs(GRID_DIR, exist_ok=True)

AUTH_KEY = "여기에_API허브_인증키"   # 재발급받은 새 키를 넣으세요 (이 노트북을 남에게 공유할 땐 지우기)

LAT, LON = 37.3921, 126.9206          # 안양대학교
URL = "https://apihub.kma.go.kr/api/typ01/url/sfc_nc_var.php"
OBS = "ta,hm,ws_10m,ta_chi"           # 기온, 습도, 풍속, 기상청 체감온도
COLS = ["tm"] + OBS.split(",")
print("프로젝트 폴더:", PROJECT_DIR)
print("저장 폴더:", GRID_DIR)

## ② 응답 확인 (처음 한 번)

한 시각만 불러서 키와 연결이 정상인지 봐요. 숫자 줄이 보이면 성공이에요.

In [ ]:
r = requests.get(URL, params={"tm1": "202501150900", "tm2": "202501150900", "obs": OBS, "itv": "60",
                             "lon": LON, "lat": LAT, "help": "0", "authKey": AUTH_KEY}, timeout=30)
print("HTTP 상태:", r.status_code)
print(r.text[:1000])

## ③ 조회 함수

- **접속을 한 번 열어두고 재사용**해요(Session). 호출할 때마다 새로 접속하지 않아서 빨라요.
- 접속 10초, 응답 30초 안에 안 오면 실패로 보고 다시 시도해요.
- `#`으로 시작하는 줄, `TM` 요약 줄은 건너뛰고 숫자로 시작하는 줄만 읽어요.
- `-999` 같은 결측값은 빈칸(NaN)으로 바꾸고, 기온이 빈 줄은 버려요.

In [ ]:
import threading
_local = threading.local()

def session():
    """스레드마다 접속 하나를 열어두고 재사용"""
    if not hasattr(_local, "s"):
        _local.s = requests.Session()
    return _local.s

def fetch(tm1: str, tm2: str, itv: int = 60) -> pd.DataFrame:
    params = {"tm1": tm1, "tm2": tm2, "obs": OBS, "itv": str(itv),
              "lon": LON, "lat": LAT, "help": "0", "authKey": AUTH_KEY}
    r = session().get(URL, params=params, timeout=(10, 30))
    r.raise_for_status()
    rows = []
    for line in r.text.splitlines():
        parts = line.strip().replace(",", " ").split()
        if len(parts) < len(COLS) or not parts[0].isdigit():
            continue
        rows.append(parts[:len(COLS)])
    df = pd.DataFrame(rows, columns=COLS)
    if df.empty:
        return df
    df["tm"] = pd.to_datetime(df["tm"], format="%Y%m%d%H%M")
    for c in COLS[1:]:
        df[c] = pd.to_numeric(df[c], errors="coerce")
        df.loc[df[c] <= -50, c] = np.nan
    return df.dropna(subset=["ta"])

t0 = time.time()
print(fetch("202501150900", "202501151000"))   # 두 시각이 함께 오는지 확인
print(f"{time.time() - t0:.1f}초")

## ④ 월별 수집 (빠른 버전)

`START`, `END`에 받을 기간(연, 월)을 적고 실행하세요.

- **한 번에 두 시각씩** 불러요 (00:00~01:00 → 두 시각이 함께 옴). 호출 수가 절반이에요.
- **동시에 4개씩** 불러요. 실패가 많으면 `WORKERS`를 2로 낮추세요.
- **진행률**을 25%마다 보여줘요.
- 실패한 구간은 최대 4번까지 다시 시도해요.
- 한 달이 끝나면 `grid_YYYY_MM.csv`로 드라이브에 저장해요. **이미 있는 달은 건너뛰어요.**
- 한 달 중 실패 구간이 10%를 넘으면 그 달은 저장하지 않고 멈춰요. 나중에 다시 실행하면 그 달부터 이어서 받아요.

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed

START = (2022, 1)    # (연, 월)
END   = (2022, 12)
WORKERS = 4          # 동시에 부를 개수

def months(start, end):
    y, m = start
    while (y, m) <= end:
        yield y, m
        y, m = (y + 1, 1) if m == 12 else (y, m + 1)

def fetch_window(t):
    """t시와 t+1시를 한 번에. 실패하면 최대 4번 다시 시도."""
    s, e = t.strftime("%Y%m%d%H%M"), (t + timedelta(hours=1)).strftime("%Y%m%d%H%M")
    for attempt in range(4):
        try:
            return fetch(s, e)
        except Exception:
            time.sleep(2 * (attempt + 1))
    return None

def fetch_month(y, m):
    start = datetime(y, m, 1)
    stop = datetime(y + (m == 12), m % 12 + 1, 1)
    windows, t = [], start
    while t < stop:
        windows.append(t)
        t += timedelta(hours=2)
    results, done, step = [], 0, max(1, len(windows) // 4)
    with ThreadPoolExecutor(max_workers=WORKERS) as ex:
        for fut in as_completed([ex.submit(fetch_window, w) for w in windows]):
            results.append(fut.result())
            done += 1
            if done % step == 0 and done < len(windows):
                print(f"   {y}-{m:02d} {done * 100 // len(windows)}%")
    failed = sum(r is None for r in results)
    if failed > len(windows) * 0.1:
        raise RuntimeError(f"{len(windows)}개 구간 중 {failed}개 실패")
    df = pd.concat([r for r in results if r is not None], ignore_index=True)
    df = df[(df["tm"] >= start) & (df["tm"] < stop)].drop_duplicates("tm").sort_values("tm")
    hours = int((stop - start).total_seconds() // 3600)
    return df, hours - len(df), failed

for y, m in months(START, END):
    path = f"{GRID_DIR}/grid_{y}_{m:02d}.csv"
    if os.path.exists(path):
        print(f"{y}-{m:02d} 이미 있음, 건너뜀")
        continue
    t0 = time.time()
    try:
        df, missing, failed = fetch_month(y, m)
    except RuntimeError as e:
        print(f"{y}-{m:02d} 멈춤:", e)
        print("한도나 서버 문제일 수 있어요. 나중에 이 칸을 다시 실행하면 이 달부터 이어서 받아요.")
        break
    df.to_csv(path, index=False, encoding="utf-8-sig")
    print(f"{y}-{m:02d} 저장: {len(df)}줄, 빈 시각 {missing}개, 실패 구간 {failed}개, {time.time() - t0:.0f}초")

## ⑤ 저장된 파일 확인

드라이브에 모인 달과 줄 수를 보여줘요. 다음 단계(데이터 합치기)는 이 폴더를 그대로 읽어요.

In [ ]:
saved = sorted(f for f in os.listdir(GRID_DIR) if f.startswith("grid_") and f.endswith(".csv"))
print(len(saved), "개월 저장됨")
summary = pd.DataFrame({"파일": saved,
                        "줄 수": [len(pd.read_csv(f"{GRID_DIR}/{f}")) for f in saved]})
summary